In [ ]:
import jax
from jax.experimental import pallas as pl
from jax.experimental.pallas import tpu as pltpu
import jax.numpy as jnp
from jax.sharding import NamedSharding, PartitionSpec as P

import numpy as np

from sfp.utils import benchmark, numerics
from sfp.kernels.sharded.distributed_gemm import jax_pallas_gemm, ag_gemm_ar_serial, fused_ag_gemm_ar

In [2]:
m, k, n = 16384, 16384, 8192

k1, k2 = jax.random.split(jax.random.key(0), 2)
lhs = jax.random.normal(k1, (m, k), dtype=jnp.bfloat16)
rhs = jax.random.normal(k2, (k, n), dtype=jnp.bfloat16)

In [ ]:
num_devices = jax.device_count()
mesh = jax.make_mesh((2, 2), ("x", "y"))
lhs_sharding = NamedSharding(mesh, P('x', 'y'))
rhs_sharding = NamedSharding(mesh, P('x', None))

lhs = jax.device_put(lhs, lhs_sharding)
rhs = jax.device_put(rhs, rhs_sharding)

In [4]:
def jax_matmul(x: jax.Array, y: jax.Array) -> jax.Array:
    return jnp.matmul(x, y)

ref = jax_matmul(lhs, rhs)

In [ ]:
jmcc = jax.jit(jax_matmul).lower(lhs, rhs).compile()
hlo_text = jmcc.as_text()
print(hlo_text)

"""
HloModule jit_jax_matmul,
is_scheduled=true,
entry_computation_layout={(bf16[8192,8192]{1,0:T(8,128)(2,1)}, bf16[8192,8192]{1,0:T(8,128)(2,1)})->bf16[8192,8192]{1,0:T(8,128)(2,1)}},
allow_spmd_sharding_propagation_to_parameters={false,false},
allow_spmd_sharding_propagation_to_output={true},
num_partitions=4

%add.clone (x.3: bf16[], y.3: bf16[]) -> bf16[] {
  %y.3 = bf16[]{:T(256)} parameter(1)
  %x.3 = bf16[]{:T(256)} parameter(0)
  ROOT %add.1 = bf16[]{:T(256)} add(%x.3, %y.3), backend_config={"flag_configs":[],"scoped_memory_configs":[],"used_scoped_memory_configs":[],"aliasing_operands":{"lists":[]}}
}

%bitcast_fusion (bitcast_input: bf16[8192,8192]) -> bf16[8192,8192] {
  %bitcast_input = bf16[8192,8192]{1,0:T(8,128)(2,1)} parameter(0)
  ROOT %bitcast = bf16[8192,8192]{1,0:T(8,128)(2,1)} bitcast(%bitcast_input)
}

%bitcast_fusion.1 (bitcast_input.1: bf16[8192,8192]) -> bf16[8192,8192] {
  %bitcast_input.1 = bf16[8192,8192]{1,0:T(8,128)(2,1)} parameter(0)
  ROOT %bitcast.1 = bf16[8192,8192]{1,0:T(8,128)(2,1)} bitcast(%bitcast_input.1)
}

%fused_computation (param_0: bf16[8192,8192], param_1: bf16[8192,8192]) -> bf16[8192,8192] {
  %param_0 = bf16[8192,8192]{1,0:T(8,128)(2,1)} parameter(0)
  %fusion.1 = bf16[8192,8192]{1,0:T(8,128)(2,1)} fusion(%param_0), kind=kLoop, calls=%bitcast_fusion
  %param_1 = bf16[8192,8192]{1,0:T(8,128)(2,1)} parameter(1)
  %fusion.2 = bf16[8192,8192]{1,0:T(8,128)(2,1)} fusion(%param_1), kind=kLoop, calls=%bitcast_fusion.1
  ROOT %convolution.1 = bf16[8192,8192]{1,0:T(8,128)(2,1)} convolution(%fusion.1, %fusion.2), dim_labels=bf_io->bf, metadata={op_name="jit(jax_matmul)/dot_general" stack_frame_id=10}
}

ENTRY %main.0_spmd (param: bf16[8192,8192], param.1: bf16[8192,8192]) -> bf16[8192,8192] {
  %param.1 = bf16[8192,8192]{1,0:T(8,128)(2,1)} parameter(1), sharding={devices=[2,1,2]<=[4] last_tile_dim_replicate}, metadata={op_name="y"}
  %param = bf16[8192,8192]{1,0:T(8,128)(2,1)} parameter(0), sharding={devices=[2,2]<=[4]}, metadata={op_name="x"}
  %collective-permute-start = (bf16[8192,8192]{1,0:T(8,128)(2,1)}, bf16[8192,8192]{1,0:T(8,128)(2,1)}, u32[]{:S(2)}, u32[]{:S(2)}) collective-permute-start(%param.1), channel_id=1, source_target_pairs={{0,0},{1,2},{2,1},{3,3}}, metadata={op_name="jit(jax_matmul)/dot_general" stack_frame_id=10}, backend_config={"flag_configs":[],"barrier_config":{"barrier_type":"CUSTOM","id":"1"},"scoped_memory_configs":[],"used_scoped_memory_configs":[]}
  %collective-permute-done = bf16[8192,8192]{1,0:T(8,128)(2,1)} collective-permute-done(%collective-permute-start), metadata={op_name="jit(jax_matmul)/dot_general" stack_frame_id=10}
  %fusion = bf16[8192,8192]{1,0:T(8,128)(2,1)} fusion(%param, %collective-permute-done), kind=kOutput, calls=%fused_computation, metadata={op_name="jit(jax_matmul)/dot_general" stack_frame_id=10}, backend_config={"flag_configs":[],"window_config":{"kernel_window_bounds":["64","8"],"output_window_bounds":["128","8"],"input_window_bounds":["128","4"],"estimated_cycles":"9557312","iteration_bounds":["8","8","16"],"cost_model_type":"COST_MODEL_TYPE_CLASSIC","ml_estimated_microseconds":0,"is_mask":false,"pad_output_on_minor_dim":"0","pad_input_on_minor_dim":"0"},"scoped_memory_configs":[],"used_scoped_memory_configs":[{"memory_space":"1","offset":"0","size":"13500416"}],"retry_config":{"retry_count":"0"},"convolution_algorithm_config":{"emitter":"EmitAllBatchInSublanes"},"aliasing_operands":{"lists":[]}}
  ROOT %all-reduce = bf16[8192,8192]{1,0:T(8,128)(2,1)} all-reduce(%fusion), channel_id=2, replica_groups=[2,2]<=[4], use_global_device_ids=true, to_apply=%add.clone, metadata={op_name="jit(jax_matmul)/dot_general" stack_frame_id=10}, backend_config={"flag_configs":[],"barrier_config":{"barrier_type":"CUSTOM","id":"0"},"scoped_memory_configs":[{"memory_space":"0","offset":"0","size":"67108864"}],"collective_algorithm_config":{"emitter":"RotatedPincerEmitter","strategy":"UniDirection1DRingStrategy","debug":"\nUniDirection1DRingStrategy{colors:2 phases:1 cores:{2},{2} nophase0:0 reserved_sflags:0 cross_module_on_2d_plane:0 has_reordering_map:0 use_routing_table_indices:0}"},"used_scoped_memory_configs":[{"memory_space":"1","offset":"0","size":"15532032"}],"retry_config":{"retry_count":"0"},"aliasing_operands":{"lists":[{"indices":["0","1"]}]}}
}
"""

In [5]:
ref

Array([[92, -233, -7, ..., -17.75, 37, -173],
       [-75, 59, 85.5, ..., -80.5, 73, -66.5],
       [53, -19.25, 24.75, ..., -58.25, -175, 206],
       ...,
       [44.75, -113, 52, ..., -116.5, 150, 8.875],
       [-73, 284, -134, ..., -8, -376, 124.5],
       [-396, -89.5, -3.21875, ..., 92.5, -27.75, 11.25]], dtype=bfloat16)

In [6]:
jax_matmul_compiled = jax.jit(jax_matmul)
jmc = jax_matmul_compiled(lhs, rhs)
jmc.block_until_ready()

with jax.profiler.trace('./traces/naive_matmul'):
    result = jax_matmul_compiled(lhs, rhs)
    result.block_until_ready()

In [8]:
benchmark(jax_matmul_compiled, lhs, rhs)

BenchmarkResult (50 iters, 3 warmup)
  mean:       12.216 ms
  median:     12.215 ms
  stdev:       0.012 ms
  min:        12.194 ms
  max:        12.258 ms
  p95:        12.237 ms
  p99:        12.252 ms

In [9]:
jpg = jax.jit(
    jax.shard_map(
        jax_pallas_gemm,
        mesh=mesh,
        in_specs=(P('x', 'y'), P('x', None)),
        out_specs=P('x', None),
        check_vma=False
    )
)

jpg_compiled = jpg.lower(lhs, rhs).compile({'xla_enable_transpose_trace': True})
result = jpg_compiled(lhs, rhs)
result.block_until_ready()

with jax.profiler.trace('./traces/jpg'):
    fc1 = jpg_compiled(lhs, rhs)
    fc1.block_until_ready()

In [23]:
benchmark(jpg_compiled, lhs, rhs)

BenchmarkResult (50 iters, 3 warmup)
  mean:       12.689 ms
  median:     12.688 ms
  stdev:       0.010 ms
  min:        12.675 ms
  max:        12.708 ms
  p95:        12.706 ms
  p99:        12.707 ms

In [ ]:
numerics.compare(ref, jpg(lhs, rhs))

In [13]:
agas = jax.jit(
    jax.shard_map(
        ag_gemm_ar_serial,
        mesh=mesh,
        in_specs=(P('x', 'y'), P('x', None)),
        out_specs=P('x', None),
        check_vma=False
    )
)

agas_compiled = agas.lower(lhs, rhs).compile({'xla_enable_transpose_trace': True})
result = agas_compiled(lhs, rhs)
result.block_until_ready()

with jax.profiler.trace('./traces/agas'):
    fc2 = jpg_compiled(lhs, rhs)
    fc2.block_until_ready()

In [22]:
benchmark(agas_compiled, lhs, rhs)

BenchmarkResult (50 iters, 3 warmup)
  mean:       13.368 ms
  median:     13.366 ms
  stdev:       0.012 ms
  min:        13.349 ms
  max:        13.407 ms
  p95:        13.390 ms
  p99:        13.402 ms

In [15]:
numerics.compare(ref, agas(lhs, rhs))

NumericsResult(FAIL)
  shape:     (16384, 8192)
  max_diff:  2.000000
  mean_diff: 0.000021
  median:    0.000000
  % > 0.1: 0.00%
  worst at (83, 2034): ref=-268.0000, test=-266.0000

In [16]:
fga = jax.jit(
    jax.shard_map(
        fused_ag_gemm_ar,
        mesh=mesh,
        in_specs=(P('x', 'y'), P('x', None)),
        out_specs=P('x', None),
        check_vma=False
    )
)

fga_compiled = fga.lower(lhs, rhs).compile({'xla_enable_transpose_trace': True})
result = fga_compiled(lhs, rhs)
result.block_until_ready()

with jax.profiler.trace('./traces/fga'):
    fc3 = fga_compiled(lhs, rhs)
    fc3.block_until_ready()

In [19]:
benchmark(fga_compiled, lhs, rhs)

BenchmarkResult (50 iters, 3 warmup)
  mean:       12.540 ms
  median:     12.543 ms
  stdev:       0.015 ms
  min:        12.511 ms
  max:        12.585 ms
  p95:        12.558 ms
  p99:        12.575 ms

In [18]:
numerics.compare(ref, fga(lhs, rhs))

NumericsResult(FAIL)
  shape:     (16384, 8192)
  max_diff:  2.000000
  mean_diff: 0.000021
  median:    0.000000
  % > 0.1: 0.00%
  worst at (83, 2034): ref=-268.0000, test=-266.0000